# Technical Assistant Planner — Agent Loop

An LLM-powered agent that takes a user's technical question, classifies intent
(**learn**, **research**, or **both**), calls the appropriate search tools in a
loop, and compiles a curated Markdown brief with web resources, videos, books,
arXiv papers, and a research landscape overview.

### Architecture

```
User prompt
    │
    ▼
┌──────────────────────────────────┐
│  System prompt (Classify →       │
│  Explain → Compile instructions) │
└──────────────┬───────────────────┘
               │
     ┌─────────▼─────────┐
     │   Agent Loop       │  ◄── repeats until no more tool calls
     │  ┌───────────────┐ │
     │  │ LLM decides   │ │
     │  │ tool calls    │ │
     │  └───────┬───────┘ │
     │          ▼         │
     │  ┌───────────────┐ │
     │  │ Execute tools │ │──► search_web, search_youtube,
     │  └───────┬───────┘ │    search_books, search_arxiv,
     │          │         │    search_research
     │  tool results back │
     │  into messages     │
     └─────────┬──────────┘
               │
               ▼
      Final Markdown brief
```

In [ ]:
from __future__ import annotations

from datetime import datetime
from enum import Enum

import logging
import os
import re
import json

import openai
import arxiv
from serpapi import GoogleSearch
from tavily import TavilyClient
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import TextFormatter
from youtube_transcript_api.proxies import WebshareProxyConfig
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from IPython.display import Markdown, display
from rich.console import Console
from rich.panel import Panel

load_dotenv(override=True)

console = Console()


import nest_asyncio
nest_asyncio.apply()


In [ ]:
openai_client = openai.OpenAI()
travily_client = TavilyClient()

## Data Models

Pydantic models for every data type flowing through the agent:
- **Classification** — intent, topic, and pre-planned search queries.
- **Tool results** — `WebResult`, `YouTubeResult`, `BookResult`, `ArxivDocument`, `ResearchReport`.
- **AgentResults** — container that collects all tool outputs for the compile step.

In [ ]:
"""Data models for the technical assistant planner."""

# ── Intent Classification ──────────────────────────────────────────


class Intent(str, Enum):
    LEARN = "learn"
    RESEARCH = "research"
    BOTH = "both"


class SearchQueries(BaseModel):
    """Pre-planned search queries generated during classification."""

    web: list[str] = Field(default_factory=list)
    youtube: list[str] = Field(default_factory=list)
    books: list[str] = Field(default_factory=list)
    arxiv: list[str] = Field(default_factory=list)


class Classification(BaseModel):
    """Result of the intent classification step."""

    intent: Intent
    topic: str
    search_queries: SearchQueries


# ── Tool Results ───────────────────────────────────────────────────


class WebResult(BaseModel):
    title: str
    url: str
    snippet: str


class YouTubeResult(BaseModel):
    title: str
    url: str
    video_id: str = ""
    channel: str = ""
    duration: str = ""  # e.g. "5:06" or "1:16:03"
    views: str = ""  # e.g. "386K views"
    published_date: str = ""  # e.g. "2 months ago"
    description: str = ""
    transcript_summary: str = ""  # LLM-generated summary of the video transcript


class BookResult(BaseModel):
    title: str
    authors: str
    description: str
    url: str
    rating: float | None = None
    price: str = ""
    category: str = ""
    source: str = ""  # "google_play" or "tavily"


class ArxivDocument(BaseModel):
    """A single arXiv paper with full metadata."""

    title: str
    authors: list[str]
    summary: str
    published: datetime
    updated: datetime
    pdf_url: str
    arxiv_url: str
    primary_category: str
    categories: list[str]
    doi: str | None = None
    comment: str | None = None
    journal_ref: str | None = None


class ArxivDocuments(BaseModel):
    """Collection of arXiv papers from a search."""

    documents: list[ArxivDocument] = Field(default_factory=list)


class ResearchReport(BaseModel):
    """Output from Tavily's deep research method."""

    report: str = ""  # The full research report text
    sources: list[WebResult] = Field(default_factory=list)


# ── Collected Results (passed to the compile step) ─────────────────


class AgentResults(BaseModel):
    """All results gathered during the tool loop."""

    classification: Classification
    topic_explanation: str = ""
    web_results: list[WebResult] = Field(default_factory=list)
    youtube_results: list[YouTubeResult] = Field(default_factory=list)
    book_results: list[BookResult] = Field(default_factory=list)
    arxiv_results: ArxivDocuments = Field(default_factory=ArxivDocuments)
    research_report: ResearchReport | None = None

## Prompt Templates & Messages

Four prompt templates, concatenated into a single system message:

| Prompt | Role |
|--------|------|
| `SYSTEM_PROMPT` | Sets the assistant's persona. |
| `CLASSIFY_PROMPT` | Instructs the LLM to classify intent and output JSON with search queries, then route to the correct tools. |
| `EXPLAIN_PROMPT` | After tools run, guides the LLM to write a topic explanation tuned to the classified intent. |
| `COMPILE_PROMPT` | Instructs the LLM to produce the final curated Markdown brief from all tool outputs. |

`USER_PROMPT` wraps the user's raw input with explicit instructions to run the full pipeline without asking follow-up questions. The `messages` list is OpenAI chat-compatible and ready to pass to the loop.

In [ ]:
"""Prompt templates for each step of the agent loop."""

SYSTEM_PROMPT = """\
You are a technical assistant planner. You help users understand \
technical topics by providing clear explanations and curated resources.\
"""

# ── Step 1: Classify ───────────────────────────────────────────────

CLASSIFY_PROMPT = """\
First, classify the user request into intent and generate targeted search queries.

Return ONLY valid JSON (no markdown, no extra prose):
{{
  "intent": "learn" | "research" | "both",
  "topic": "<concise topic name>",
  "search_queries": {{
    "web": ["<query1>", "<query2>"],
    "youtube": ["<query1>", "<query2>"],
    "books": ["<query1>", "<query2>"],
    "arxiv": ["<query1>", "<query2>"]
  }}
}}

Routing rules for tool calls after classification:
- learn -> call: search_web, search_youtube, search_books
- research -> call: search_research and search_arxiv
- both -> call all five tools

Query generation rules:
- Generate 2-3 specific queries per relevant key.
- Use only intent-relevant keys in search_queries.
- Keep queries high-signal and non-redundant.
- Never ask follow-up or clarifying questions. Proceed with best assumptions.\
"""

# ── Step 2: Explain ────────────────────────────────────────────────

EXPLAIN_PROMPT = """\
After classification and tool execution, write a concise topic explanation.

Inputs:
- topic: {topic}
- user_input: {user_input}
- intent: {intent}
- optional evidence from tools (web/youtube/books/arxiv/research)

Rules:
- learn: explain fundamentals in plain language with practical intuition.
- research: summarize current research landscape, active directions, and open questions.
- both: briefly cover fundamentals, then bridge into research directions.
- Keep it concise (about 150-300 words), factual, and directly useful.
- Do not ask clarifying questions.\
"""

# ── Step 3: Compile ────────────────────────────────────────────────

COMPILE_PROMPT = """\
Create the final Markdown response using gathered tool outputs.

Inputs:
Topic: {topic}
Intent: {intent}
Topic explanation: {explanation}
Web results: {web_results}
YouTube results: {youtube_results}
Book results: {book_results}
arXiv results: {arxiv_results}
Research report: {research_report}

Output requirements:
- Start with: # {topic}
- Always include: ## Summary (use and refine the explanation)
- If intent includes learn, include:
  - ## Web Resources
  - ## Video Resources
  - ## Book Recommendations
- If intent includes research, include:
  - ## Research Landscape
  - ## Key Papers
- End with: ## Practical Next Steps (3-5 actions)

Formatting rules:
- Curate quality over quantity.
- Include only resources present in tool outputs.
- Add one short "why useful" note per resource.
- Output raw Markdown only.
- Never ask follow-up questions.\
"""

# ── Runtime user message and OpenAI-compatible messages ────────────

USER_PROMPT = """\
User request:
{user_input}

Handle this end-to-end in one run:
1) Classify intent.
2) Call the appropriate tools based on that intent.
3) Explain the topic.
4) Compile a final curated Markdown response.

Do not ask follow-up or clarifying questions. Make reasonable assumptions and proceed.\
"""

# ── Transcript Summary ─────────────────────────────────────────────

TRANSCRIPT_SUMMARY_PROMPT = """\
Summarize this YouTube video transcript in the context of the topic below.

Topic: {topic}
Video title: {title}
Channel: {channel}

Transcript:
{transcript}

Rules:
- Write 2-4 sentences summarizing the key takeaways relevant to the topic.
- Focus on what a learner would find most useful from this video.
- If the transcript is mostly promotional or off-topic, say so briefly.
- Be concise and direct.\
"""


def input_messages(user_input: str) -> str:
  # Set this before starting the loop, for example:
  # user_input = "I want to learn retrieval augmented generation and latest RAG research"
  messages = [
      {
          "role": "system",
          "content": "\n\n".join([
              SYSTEM_PROMPT,
              CLASSIFY_PROMPT,
              EXPLAIN_PROMPT,
              COMPILE_PROMPT,
          ]),
      },
      {
          "role": "user",
          "content": USER_PROMPT.format(user_input=user_input),
      },
  ]
  return messages

## Tool Implementations

The actual Python functions the agent can invoke. Each tool accepts query lists (and optional config) and returns typed Pydantic results.

| Function | Sources | Returns |
|----------|---------|---------|
| `search_web` | Tavily | `list[WebResult]` |
| `search_youtube` | SerpAPI (primary) + Tavily (fallback), with optional transcript summarization via OpenAI | `list[YouTubeResult]` |
| `search_books` | Google Play Books via SerpAPI + Tavily (Goodreads, Amazon) | `list[BookResult]` |
| `search_arxiv` | arXiv API | `ArxivDocuments` |
| `search_research` | Tavily deep search + arXiv | `(ResearchReport, ArxivDocuments)` |

Private helpers (`_tavily_search`, `_fetch_transcript`, `_summarize_transcript`, etc.) are defined here too but are not exposed as agent tools.

In [ ]:
"""Tool implementations for the agent.

Each tool takes a list of queries, runs them, and returns structured results.
- Tavily handles web search and deep research.
- SerpAPI handles YouTube search and Google Play Books search.
- youtube-transcript-api fetches video transcripts for LLM summarization.
- Tavily provides fallback for both YouTube and books.
- The arxiv library handles paper search.
"""

logger = logging.getLogger(__name__)


def _tavily_search(
    queries: list[str],
    *,
    include_domains: list[str] | None = None,
    max_results: int = 5,
) -> list[dict]:
    """Run multiple queries through Tavily and merge results."""
    seen_urls: set[str] = set()
    results: list[dict] = []

    for query in queries:
        response = travily_client.search(
            query=query,
            max_results=max_results,
            include_domains=include_domains or [],
        )
        for item in response.get("results", []):
            if item["url"] not in seen_urls:
                seen_urls.add(item["url"])
                results.append(item)

    return results


# ── Web Search (tutorials, blogs, docs) ────────────────────────────


def search_web(queries: list[str]) -> list[WebResult]:
    """Search for tutorials, blog posts, and documentation."""
    console.print(Panel(
        "\n".join(f"  [cyan]•[/] {q}" for q in queries),
        title="[bold blue]search_web[/]",
        subtitle=f"{len(queries)} queries",
        border_style="blue",
    ))
    raw = _tavily_search(queries, max_results=5)
    results = [
        WebResult(
            title=r.get("title", ""),
            url=r["url"],
            snippet=r.get("content", "")[:2000],
        )
        for r in raw
    ]
    console.print(f"  [green]✓[/] Found [bold]{len(results)}[/] web results\n")
    return results


# ── YouTube Search (SerpAPI primary + Tavily fallback) ─────────────


class YouTubeProxySettings:
    """Settings for YouTube transcript proxy configuration.

    Reads from environment variables:
      YOUTUBE_PROXY_ENABLED=true
      YOUTUBE_PROXY_USERNAME=your-username
      YOUTUBE_PROXY_PASSWORD=your-password
    """

    def __init__(self) -> None:
        self.enabled = os.getenv("YOUTUBE_PROXY_ENABLED", "false").lower() == "true"
        self.username = os.getenv("YOUTUBE_PROXY_USERNAME", "")
        self.password = os.getenv("YOUTUBE_PROXY_PASSWORD", "")

    @property
    def is_configured(self) -> bool:
        return self.enabled and bool(self.username) and bool(self.password)


# Module-level instance — loaded once
_proxy_settings = YouTubeProxySettings()


def _fetch_transcript(
    video_id: str,
    *,
    languages: list[str] | None = None,
) -> str | None:
    """Fetch a YouTube video transcript. Returns the full text or None on failure.

    Supports optional Webshare proxy for environments where YouTube
    blocks transcript requests (e.g. cloud servers).
    """
    if languages is None:
        languages = ["en"]

    try:
        if _proxy_settings.is_configured:
            proxy_config = WebshareProxyConfig(
                proxy_username=_proxy_settings.username,
                proxy_password=_proxy_settings.password,
            )
            ytt_api = YouTubeTranscriptApi(proxy_config=proxy_config)
            logger.info("Using Webshare proxy for transcript: %s", video_id)
        else:
            ytt_api = YouTubeTranscriptApi()

        fetched = ytt_api.fetch(video_id, languages=languages)
        formatter = TextFormatter()
        full_text = formatter.format_transcript(fetched)

        logger.info("Transcript fetched for %s (%d chars)", video_id, len(full_text))
        return full_text

    except Exception as e:
        print(f"Failed to fetch transcript for {video_id}: {e}")
        logger.debug("Failed to fetch transcript for %s", video_id, exc_info=True)
        return None


def _summarize_transcript(
    *,
    transcript: str,
    topic: str,
    title: str,
    channel: str,
) -> str:
    """Use the LLM to summarize a video transcript in context of the topic."""
    # Cap transcript to ~6000 chars to stay within reasonable token usage
    # trimmed = transcript[:6000]
    client: openai.OpenAI = openai_client
    prompt = TRANSCRIPT_SUMMARY_PROMPT.format(
        topic=topic,
        title=title,
        channel=channel,
        transcript=transcript,
    )

    # message = client.messages.create(
    #     model="claude-sonnet-4-20250514",
    #     max_tokens=256,
    #     system=SYSTEM_PROMPT,
    #     messages=[{"role": "user", "content": prompt}],
    # )
    # return message.content[0].text
    response = client.chat.completions.create(
        model="gpt-4.1-mini", 
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}])
    transcript_summary = response.choices[0].message.content
    # print(f"transcript_summary: {transcript_summary}")
    return transcript_summary


def _enrich_with_transcripts(
    results: list[YouTubeResult],
    topic: str,
    *,
    max_videos: int = 5,
) -> list[YouTubeResult]:
    """Fetch transcripts and generate summaries for the top N videos.

    Only processes the first max_videos results to keep API costs
    and latency reasonable. Videos without available transcripts
    are left with an empty transcript_summary.
    """
    client: openai.OpenAI = openai_client
    for video in results[:max_videos]:
        # Extract video_id from URL
        # video_id = ""
        # if "v=" in video.url:
        #     video_id = video.url.split("v=")[-1].split("&")[0]
        # elif "youtu.be/" in video.url:
        #     video_id = video.url.split("youtu.be/")[-1].split("?")[0]
        video_id = video.video_id

        if not video_id:
            continue

        transcript = _fetch_transcript(video_id, languages=["en"])
        # print(f"transcript: {transcript}")
        if not transcript:
            continue

        try:
            video.transcript_summary = _summarize_transcript(
                transcript=transcript,
                topic=topic,
                title=video.title,
                channel=video.channel,
            )
        except Exception as e:
            print(f"Failed to summarize transcript for {video_id}: {e}")
            logger.debug("Failed to summarize transcript for %s", video_id)

    return results


def _parse_relative_date_to_months(date_str: str) -> float | None:
    """Parse SerpAPI's relative date strings into approximate months ago.

    Handles formats like: "3 days ago", "2 months ago", "1 year ago",
    "Streamed 2 months ago", "18 hours ago".
    Returns None if the format is unrecognized.
    """
    # Strip "Streamed " prefix if present
    cleaned = date_str.replace("Streamed ", "").strip().lower()

    match = re.match(r"(\d+)\s+(hour|day|week|month|year)s?\s+ago", cleaned)
    if not match:
        return None

    amount = int(match.group(1))
    unit = match.group(2)

    multipliers = {
        "hour": 1 / 720,  # ~720 hours per month
        "day": 1 / 30,
        "week": 1 / 4.3,
        "month": 1,
        "year": 12,
    }
    return amount * multipliers.get(unit, 0)


def _format_views_count(views: int) -> str:
    """Format an integer view count into a human-readable string."""
    if views >= 1_000_000:
        return f"{views / 1_000_000:.1f}M views"
    if views >= 1_000:
        return f"{views / 1_000:.1f}K views"
    return f"{views} views"


def _is_relevant(video: dict, query_keywords: set[str], min_keyword_hits: int = 1) -> bool:
    """Check if a video is relevant to the search query by keyword matching.

    Looks for query keywords in the video title and description.
    """
    text = (video.get("title", "") + " " + video.get("description", "")).lower()
    hits = sum(1 for kw in query_keywords if kw in text)
    return hits >= min_keyword_hits


def _search_youtube_serpapi(
    queries: list[str],
    serp_api_key: str,
    *,
    max_age_months: int = 12,
    max_results: int = 8,
) -> list[YouTubeResult]:
    """Search YouTube via SerpAPI, filter by recency and relevance."""
    seen_ids: set[str] = set()
    results: list[YouTubeResult] = []

    for query in queries:
        # Build keyword set for relevance filtering
        query_keywords = {w.lower() for w in query.split() if len(w) > 2}

        search = GoogleSearch({
            "api_key": serp_api_key,
            "engine": "youtube",
            "search_query": query,
        })
        raw = search.get_dict()

        for video in raw.get("video_results", []):
            video_id = video.get("video_id", "")
            if not video_id or video_id in seen_ids:
                continue

            # Recency filter — skip videos older than max_age_months
            published = video.get("published_date", "")
            months_ago = _parse_relative_date_to_months(published)
            if months_ago is not None and months_ago > max_age_months:
                continue

            # Relevance filter — title or description must match query keywords
            if not _is_relevant(video, query_keywords):
                continue

            seen_ids.add(video_id)
            channel = video.get("channel", {})
            results.append(
                YouTubeResult(
                    title=video.get("title", ""),
                    url=video.get("link", f"https://www.youtube.com/watch?v={video_id}"),
                    video_id=video_id,
                    channel=channel.get("name", ""),
                    duration=video.get("length", ""),
                    views=_format_views_count(video.get("views", 0)),
                    published_date=published,
                    description=video.get("description", ""),
                )
            )

    # Sort by views (highest first) and cap results
    results.sort(key=lambda v: v.views, reverse=True)
    return results[:max_results]


def _search_youtube_tavily(
    queries: list[str]
) -> list[YouTubeResult]:
    """Fallback YouTube search using Tavily with domain filtering."""
    raw = _tavily_search(
        queries,
        include_domains=["youtube.com"],
        max_results=5,
    )
    return [
        YouTubeResult(
            title=r.get("title", ""),
            url=r["url"],
            description=r.get("content", "")[:200],
        )
        for r in raw
    ]


def search_youtube(
    queries: list[str],
    *,
    serp_api_key: str | None = None,
    topic: str = "",
) -> list[YouTubeResult]:
    """Search YouTube: SerpAPI for rich metadata + filtering, Tavily as fallback."""
    console.print(Panel(
        "\n".join(f"  [cyan]•[/] {q}" for q in queries),
        title="[bold red]search_youtube[/]",
        subtitle=f"topic={topic!r}" if topic else f"{len(queries)} queries",
        border_style="red",
    ))
    results: list[YouTubeResult] = []

    if serp_api_key:
        try:
            results = _search_youtube_serpapi(queries, serp_api_key)
            console.print(f"  [dim]SerpAPI returned {len(results)} videos[/]")
        except Exception:
            console.print("  [yellow]SerpAPI failed — falling back to Tavily[/]")

    if not results:
        results = _search_youtube_tavily(queries)
        console.print(f"  [dim]Tavily fallback returned {len(results)} videos[/]")

    if results and openai_client and topic:
        console.print(f"  [dim]Enriching top videos with transcript summaries…[/]")
        results = _enrich_with_transcripts(results, topic)

    console.print(f"  [green]✓[/] Found [bold]{len(results)}[/] YouTube results\n")
    return results


# ── Book Search (SerpAPI Google Play Books + Tavily) ───────────────


def _search_google_play_books(
    query: str,
    serp_api_key: str,
    *,
    gl: str = "us",
    hl: str = "en",
) -> list[BookResult]:
    """Search Google Play Books via SerpAPI and return structured results."""
    params = {
        "api_key": serp_api_key,
        "engine": "google_play_books",
        "q": query,
        "gl": gl,
        "hl": hl,
    }
    search = GoogleSearch(params)
    raw = search.get_dict()

    results: list[BookResult] = []
    for section in raw.get("organic_results", []):
        for book in section.get("items", []):
            rating = book.get("rating")
            results.append(
                BookResult(
                    title=book.get("title", "Untitled"),
                    authors=book.get("author", "Unknown author"),
                    description=book.get("description", "")[:300],
                    url=book.get("link", ""),
                    rating=float(rating) if rating and rating > 0 else None,
                    price=book.get("price", "N/A"),
                    category=book.get("category", ""),
                    source="google_play",
                )
            )

    return results


def _search_books_tavily(
    queries: list[str]
) -> list[BookResult]:
    """Fallback book search using Tavily against Goodreads and Amazon."""
    book_queries = [f"best books about {q}" for q in queries]
    raw = _tavily_search(
        book_queries,
        include_domains=["goodreads.com", "amazon.com"],
        max_results=5,
    )
    return [
        BookResult(
            title=r.get("title", ""),
            authors="",
            description=r.get("content", "")[:200],
            url=r["url"],
            source="tavily",
        )
        for r in raw
    ]


def search_books(
    queries: list[str],
    serp_api_key: str | None = None,
) -> list[BookResult]:
    """Search for books: Google Play Books (SerpAPI) + Tavily for broader coverage."""
    console.print(Panel(
        "\n".join(f"  [cyan]•[/] {q}" for q in queries),
        title="[bold magenta]search_books[/]",
        subtitle=f"{len(queries)} queries",
        border_style="magenta",
    ))
    results: list[BookResult] = []
    seen_titles: set[str] = set()

    if serp_api_key:
        for query in queries:
            try:
                play_results = _search_google_play_books(query, serp_api_key)
                for book in play_results:
                    title_key = book.title.lower().strip()
                    if title_key not in seen_titles:
                        seen_titles.add(title_key)
                        results.append(book)
            except Exception:
                pass
        if results:
            console.print(f"  [dim]Google Play returned {len(results)} books[/]")

    tavily_results = _search_books_tavily(queries)
    added = 0
    for book in tavily_results:
        title_key = book.title.lower().strip()
        if title_key not in seen_titles:
            seen_titles.add(title_key)
            results.append(book)
            added += 1
    if added:
        console.print(f"  [dim]Tavily added {added} more books[/]")

    console.print(f"  [green]✓[/] Found [bold]{len(results)}[/] book results\n")
    return results


# ── arXiv Search ───────────────────────────────────────────────────


def search_arxiv(queries: list[str], max_results: int = 5) -> ArxivDocuments:
    """Search arXiv for papers across multiple queries with full metadata."""
    console.print(Panel(
        "\n".join(f"  [cyan]•[/] {q}" for q in queries),
        title="[bold yellow]search_arxiv[/]",
        subtitle=f"max_results={max_results}",
        border_style="yellow",
    ))
    seen_ids: set[str] = set()
    documents: list[ArxivDocument] = []

    arxiv_client = arxiv.Client()

    for query in queries:
        search = arxiv.Search(
            query=query,
            max_results=max_results,
            sort_by=arxiv.SortCriterion.Relevance,
        )
        for result in arxiv_client.results(search):
            if result.entry_id not in seen_ids:
                seen_ids.add(result.entry_id)
                documents.append(
                    ArxivDocument(
                        title=result.title,
                        authors=[a.name for a in result.authors],
                        summary=result.summary,
                        published=result.published,
                        updated=result.updated,
                        pdf_url=result.pdf_url,
                        arxiv_url=result.entry_id,
                        primary_category=result.primary_category,
                        categories=result.categories,
                        doi=result.doi,
                        comment=result.comment,
                        journal_ref=result.journal_ref,
                    )
                )

    console.print(f"  [green]✓[/] Found [bold]{len(documents)}[/] arXiv papers\n")
    return ArxivDocuments(documents=documents)


# ── Research (Tavily deep research + arXiv) ────────────────────────


def search_research(
    topic: str, queries: list[str]
) -> tuple[ResearchReport, ArxivDocuments]:
    """Run Tavily deep research for a synthesized report, plus arXiv for papers."""
    console.print(Panel(
        f"  [cyan]Topic:[/] {topic}\n" + "\n".join(f"  [cyan]•[/] {q}" for q in queries),
        title="[bold green]search_research[/]",
        subtitle="Tavily + arXiv",
        border_style="green",
    ))

    tavily: TavilyClient = travily_client
    raw = tavily.search(topic)

    report = ResearchReport(
        report=raw.get("query", ""),
        sources=[
            WebResult(
                title=s.get("title", ""),
                url=s.get("url", ""),
                snippet=s.get("content", s.get("title", "")),
            )
            for s in raw.get("results", [])
        ],
    )
    console.print(f"  [dim]Tavily report gathered {len(report.sources)} sources[/]")

    arxiv_results = search_arxiv(queries)

    console.print(f"  [green]✓[/] Research complete — [bold]{len(report.sources)}[/] sources + [bold]{len(arxiv_results.documents)}[/] papers\n")
    return report, arxiv_results

## Tool Schemas (OpenAI function calling format)

JSON schemas that describe each tool to the LLM, following the OpenAI `tools` parameter format. The `tools` list at the bottom is what gets passed to `chat.completions.create()`.

In [ ]:
"""OpenAI-compatible function tool schemas for the planner agent loop."""

search_web_json = {
    "name": "search_web",
    "description": "Search the web for tutorials, blog posts, documentation, and articles relevant to the user's topic.",
    "parameters": {
        "type": "object",
        "properties": {
            "queries": {
                "type": "array",
                "items": {"type": "string"},
                "description": "One or more specific search queries (e.g. from classification search_queries.web).",
            },
        },
        "required": ["queries"],
        "additionalProperties": False,
    },
}

search_youtube_json = {
    "name": "search_youtube",
    "description": "Search YouTube for educational videos. Uses SerpAPI when configured for rich metadata; may enrich top results with transcript summaries when a topic is provided.",
    "parameters": {
        "type": "object",
        "properties": {
            "queries": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Search queries for videos (e.g. from classification search_queries.youtube).",
            },
            "topic": {
                "type": "string",
                "description": "Classified topic string for transcript summarization; omit or leave empty to skip LLM transcript summaries.",
            },
        },
        "required": ["queries"],
        "additionalProperties": False,
    },
}

search_books_json = {
    "name": "search_books",
    "description": "Search for books via Google Play (SerpAPI) and supplemental web sources (Goodreads, Amazon via Tavily).",
    "parameters": {
        "type": "object",
        "properties": {
            "queries": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Book-focused queries (e.g. from classification search_queries.books).",
            },
        },
        "required": ["queries"],
        "additionalProperties": False,
    },
}

search_arxiv_json = {
    "name": "search_arxiv",
    "description": "Search arXiv for academic papers: titles, abstracts, authors, PDF and arXiv URLs, categories.",
    "parameters": {
        "type": "object",
        "properties": {
            "queries": {
                "type": "array",
                "items": {"type": "string"},
                "description": "arXiv search strings (e.g. from classification search_queries.arxiv).",
            },
            "max_results": {
                "type": "integer",
                "description": "Maximum papers to fetch per query; default 5 if omitted.",
            },
        },
        "required": ["queries"],
        "additionalProperties": False,
    },
}

search_research_json = {
    "name": "search_research",
    "description": "Deeper research: synthesize a report from Tavily web search and fetch related arXiv papers for the same theme.",
    "parameters": {
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "Main research question or theme for Tavily (passed to search_research as topic).",
            },
            "queries": {
                "type": "array",
                "items": {"type": "string"},
                "description": "arXiv search queries aligned with the research topic.",
            },
        },
        "required": ["topic", "queries"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": search_web_json},
    {"type": "function", "function": search_youtube_json},
    {"type": "function", "function": search_books_json},
    {"type": "function", "function": search_arxiv_json},
    {"type": "function", "function": search_research_json},
]


## Agent Loop

Core execution engine:

1. **`handle_tool_calls`** — dispatches each tool call from the LLM response, serializes Pydantic results to JSON, and returns OpenAI-compatible tool-role messages.
2. **`loop`** — repeatedly calls the LLM with accumulated messages + tools until the model stops requesting tool calls, then renders the final Markdown output.

Rich console output shows each step, tool dispatch, and result counts as the loop progresses.

In [ ]:
def show(content):
    display(Markdown(content))


def _serialize(obj):
    """Convert tool return values to JSON-safe dicts/lists."""
    if isinstance(obj, BaseModel):
        return obj.model_dump(mode="json")
    if isinstance(obj, (list, tuple)):
        return [_serialize(item) for item in obj]
    if isinstance(obj, dict):
        return {k: _serialize(v) for k, v in obj.items()}
    return obj


def handle_tool_calls(tool_calls):
    results = []
    console.print(f"\n[bold cyan]Agent requesting {len(tool_calls)} tool call(s)[/]\n")
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        console.print(f"  [dim]→ dispatching[/] [bold]{tool_name}[/]")
        tool = globals().get(tool_name)
        if tool is None:
            console.print(f"  [bold red]✗ Unknown tool:[/] {tool_name}")
            result = {"error": f"unknown tool {tool_name}"}
        else:
            result = tool(**arguments)
        results.append({"role": "tool", "content": json.dumps(_serialize(result)), "tool_call_id": tool_call.id})
    return results


def loop(messages):
    console.rule("[bold]Agent Loop Started[/]", style="bright_blue")
    step = 0
    done = False
    while not done:
        step += 1
        console.print(f"\n[bold bright_blue]Step {step}[/] — calling LLM…")
        response = openai_client.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    console.rule("[bold green]Agent Loop Complete[/]", style="green")
    show(response.choices[0].message.content)

## Tool Smoke Tests

Individual calls to each tool to verify they work before running the full agent loop. These cells are for development/debugging — skip them during normal use.

In [ ]:
web_search_results = search_web(queries=["machine learning", "AI engineering"])
web_search_results

In [ ]:
youtube_search_results = search_youtube(
    queries=["machine learning", "AI engineering"], 
    serp_api_key=os.getenv("SERP_API_KEY"),
    topic="machine learning and AI engineering"
)
youtube_search_results

In [ ]:
book_search_results = search_books(queries=["machine learning", "AI engineering"], serp_api_key=os.getenv("SERP_API_KEY"))
print(f"Search results: {len(book_search_results)}")
book_search_results[:10]

In [ ]:
aarxiv_docs = search_arxiv(["machine learning", "AI engineering"])
arxiv_docs = aarxiv_docs.documents
print(f"arXiv docs: {len(arxiv_docs)}")
print(f"arXiv doc: {arxiv_docs[0]}")
print(f"arXiv doc categories: {arxiv_docs[0].categories}")
print(f"arXiv doc primary category: {arxiv_docs[0].primary_category}")
print(f"arXiv doc journal ref: {arxiv_docs[0].journal_ref}")


In [ ]:
reasearch_result = search_research(topic="machine learning and AI engineering", queries=["machine learning", "AI engineering"])

In [ ]:
reasearch_result

In [ ]:
# _serialize(reasearch_result)

## Run the Agent

Set `user_input` and call `loop(input_messages(user_input))`. The agent will classify intent, call the appropriate tools, and produce a curated Markdown brief — all visible in the rich console output above each final result.

Each cell below demonstrates a different intent type:
- **Learn** — concept explanation + web/video/book resources.
- **Research** — arXiv papers + Tavily research report.
- **Both** — full pipeline across all tools.

In [ ]:
user_input = "I want to learn retrieval augmented generation and latest RAG research"
messages = input_messages(user_input)
loop(messages)


In [ ]:
user_input = "I am conducting research on Speect to speech translation. Which are the latest research papers on the topic?"
messages = input_messages(user_input)
loop(messages)

In [ ]:
user_input = "I want to develop a restful api with python and django using django rest framework. \
    What are the best practices for the project structure? What study resources can you recommend?"
messages = input_messages(user_input)
loop(messages)

In [ ]:
user_input = "I am learning about application containerization. Recommend study resources on docker and kubernetes"
messages = input_messages(user_input)
loop(messages)